In [10]:
from jetbot import Camera

camera = Camera.instance(
    width=224,
    height=224
)

print("Camera ready!")

Camera ready!


In [2]:
import os

far_dir = "datasets/stop/far"
close_dir = "datasets/stop/close"

os.makedirs(far_dir, exist_ok=True)
os.makedirs(close_dir, exist_ok=True)

print("STOP dataset folders ready!")
print("FAR:", far_dir)
print("CLOSE:", close_dir)

STOP dataset folders ready!
FAR: datasets/stop/far
CLOSE: datasets/stop/close


In [4]:
none_dir = "datasets/stop/none"
os.makedirs(none_dir, exist_ok=True)

print("NONE:", none_dir)

NONE: datasets/stop/none


In [7]:
import os
import time
import threading
import ipywidgets as widgets

from IPython.display import display
from jetbot import Robot, bgr8_to_jpeg
from traitlets import dlink


# ============================================================
# DATASET FOLDERS
# ============================================================

none_dir = "datasets/stop/none"
far_dir = "datasets/stop/far"
close_dir = "datasets/stop/close"

os.makedirs(none_dir, exist_ok=True)
os.makedirs(far_dir, exist_ok=True)
os.makedirs(close_dir, exist_ok=True)

CLASS_FOLDERS = {
    "NONE": none_dir,
    "FAR": far_dir,
    "CLOSE": close_dir
}


# ============================================================
# ROBOT
# ============================================================

robot = Robot()

LEFT_GAIN = 1.035
RIGHT_GAIN = 1.00

STARTUP_SPEED = 0.18
STARTUP_TIME = 0.15

drive_direction = 0


# ============================================================
# CAMERA VIEW
# ============================================================

camera_view = widgets.Image(
    format="jpeg",
    width=300,
    height=300
)

camera_link = dlink(
    (camera, "value"),
    (camera_view, "value"),
    transform=bgr8_to_jpeg
)


# ============================================================
# DRIVE SLIDERS
# ============================================================

speed_slider = widgets.FloatSlider(
    value=0.09,
    min=0.05,
    max=0.30,
    step=0.01,
    description="Speed:"
)

steering = widgets.FloatSlider(
    value=0.0,
    min=-1.0,
    max=1.0,
    step=0.01,
    description="Steering:"
)


# ============================================================
# DRIVE BUTTONS
# ============================================================

start_button = widgets.Button(description="START")
stop_button = widgets.Button(description="STOP")
back_button = widgets.Button(description="BACK")


# ============================================================
# MOTOR CONTROL
# ============================================================

def update_motors(change=None):

    global drive_direction

    if drive_direction == 0:
        robot.stop()
        return

    speed = speed_slider.value
    s = steering.value

    steering_power = s * 0.10

    left_speed = (speed + steering_power) * LEFT_GAIN
    right_speed = (speed - steering_power) * RIGHT_GAIN

    left_speed *= drive_direction
    right_speed *= drive_direction

    left_speed = max(-1.0, min(1.0, left_speed))
    right_speed = max(-1.0, min(1.0, right_speed))

    robot.left_motor.value = left_speed
    robot.right_motor.value = right_speed


def start_drive(b):

    global drive_direction

    drive_direction = 1

    robot.left_motor.value = STARTUP_SPEED * LEFT_GAIN
    robot.right_motor.value = STARTUP_SPEED * RIGHT_GAIN

    time.sleep(STARTUP_TIME)

    update_motors()

    print("FORWARD")


def stop_drive(b):

    global drive_direction

    drive_direction = 0

    robot.stop()

    print("STOPPED")


def back_drive(b):

    global drive_direction

    drive_direction = -1

    update_motors()

    print("BACKWARD")


start_button.on_click(start_drive)
stop_button.on_click(stop_drive)
back_button.on_click(back_drive)

speed_slider.observe(update_motors, names="value")
steering.observe(update_motors, names="value")


# ============================================================
# IMAGE FUNCTIONS
# ============================================================

def get_images(folder):

    return sorted([
        f for f in os.listdir(folder)
        if f.lower().endswith(
            (".jpg", ".jpeg", ".png")
        )
    ])


def save_image(folder):

    filename = (
        str(int(time.time() * 1000))
        + ".jpg"
    )

    path = os.path.join(
        folder,
        filename
    )

    with open(path, "wb") as f:
        f.write(
            bgr8_to_jpeg(
                camera.value
            )
        )

    update_counts()

    return filename


# ============================================================
# COUNTERS
# ============================================================

none_count = widgets.IntText(
    description="NONE:",
    disabled=True
)

far_count = widgets.IntText(
    description="FAR:",
    disabled=True
)

close_count = widgets.IntText(
    description="CLOSE:",
    disabled=True
)


def update_counts():

    none_count.value = len(
        get_images(none_dir)
    )

    far_count.value = len(
        get_images(far_dir)
    )

    close_count.value = len(
        get_images(close_dir)
    )


update_counts()


# ============================================================
# MANUAL SAVE
# ============================================================

save_none_button = widgets.Button(
    description="SAVE NONE"
)

save_far_button = widgets.Button(
    description="SAVE FAR"
)

save_close_button = widgets.Button(
    description="SAVE CLOSE"
)


def save_none(b):

    filename = save_image(none_dir)

    print(
        "Saved NONE:",
        filename
    )


def save_far(b):

    filename = save_image(far_dir)

    print(
        "Saved FAR:",
        filename
    )


def save_close(b):

    filename = save_image(close_dir)

    print(
        "Saved CLOSE:",
        filename
    )


save_none_button.on_click(save_none)
save_far_button.on_click(save_far)
save_close_button.on_click(save_close)


# ============================================================
# AUTO RECORD
# ============================================================

RECORD_INTERVAL = 0.20

recording = False
record_class = None


record_none_button = widgets.Button(
    description="RECORD NONE"
)

record_far_button = widgets.Button(
    description="RECORD FAR"
)

record_close_button = widgets.Button(
    description="RECORD CLOSE"
)

stop_record_button = widgets.Button(
    description="STOP RECORD"
)

record_status = widgets.Label(
    value="Recording: OFF"
)


def record_loop():

    global recording
    global record_class

    while recording:

        # שומר רק בזמן שהרובוט נוסע
        if drive_direction != 0:

            folder = CLASS_FOLDERS[
                record_class
            ]

            save_image(folder)

        time.sleep(
            RECORD_INTERVAL
        )


def start_recording(selected_class):

    global recording
    global record_class

    recording = False

    time.sleep(0.05)

    record_class = selected_class
    recording = True

    record_status.value = (
        "Recording: "
        + selected_class
    )

    threading.Thread(
        target=record_loop,
        daemon=True
    ).start()

    print(
        "RECORDING",
        selected_class
    )


def record_none(b):

    start_recording(
        "NONE"
    )


def record_far(b):

    start_recording(
        "FAR"
    )


def record_close(b):

    start_recording(
        "CLOSE"
    )


def stop_recording(b):

    global recording
    global record_class

    recording = False
    record_class = None

    record_status.value = (
        "Recording: OFF"
    )

    print(
        "RECORDING STOPPED"
    )


record_none_button.on_click(
    record_none
)

record_far_button.on_click(
    record_far
)

record_close_button.on_click(
    record_close
)

stop_record_button.on_click(
    stop_recording
)


# ============================================================
# DELETE CLASS SELECTOR
# ============================================================

class_selector = widgets.Dropdown(
    options=[
        "NONE",
        "FAR",
        "CLOSE"
    ],
    value="NONE",
    description="Class:"
)


# ============================================================
# DELETE LAST
# ============================================================

delete_last_button = widgets.Button(
    description="DELETE LAST"
)


def delete_last(b):

    if recording:

        print(
            "Stop RECORD before deleting."
        )

        return

    selected_class = (
        class_selector.value
    )

    folder = CLASS_FOLDERS[
        selected_class
    ]

    images = get_images(folder)

    if len(images) == 0:

        print(
            selected_class,
            "folder is empty."
        )

        return

    filename = images[-1]

    os.remove(
        os.path.join(
            folder,
            filename
        )
    )

    update_counts()

    print(
        "Deleted:",
        selected_class,
        filename
    )


delete_last_button.on_click(
    delete_last
)


# ============================================================
# DELETE RANGE
# ============================================================

from_box = widgets.IntText(
    value=1,
    description="From:"
)

to_box = widgets.IntText(
    value=1,
    description="To:"
)

delete_range_button = widgets.Button(
    description="DELETE RANGE"
)


def delete_range(b):

    if recording:

        print(
            "Stop RECORD before deleting."
        )

        return

    selected_class = (
        class_selector.value
    )

    folder = CLASS_FOLDERS[
        selected_class
    ]

    images = get_images(folder)

    start = from_box.value
    end = to_box.value

    if start < 1:

        print(
            "From must be >= 1"
        )

        return

    if end < start:

        print(
            "To must be >= From"
        )

        return

    if start > len(images):

        print(
            "Start number does not exist."
        )

        return

    end = min(
        end,
        len(images)
    )

    images_to_delete = images[
        start - 1:end
    ]

    for filename in images_to_delete:

        os.remove(
            os.path.join(
                folder,
                filename
            )
        )

    update_counts()

    print(
        "Deleted",
        len(images_to_delete),
        "images from",
        selected_class,
        "range",
        start,
        "-",
        end
    )


delete_range_button.on_click(
    delete_range
)


# ============================================================
# DELETE SPECIFIC IMAGE
# ============================================================

image_input = widgets.Text(
    description="Image:",
    placeholder="number or filename"
)

delete_image_button = widgets.Button(
    description="DELETE IMAGE"
)


def delete_specific_image(b):

    if recording:

        print(
            "Stop RECORD before deleting."
        )

        return

    selected_class = (
        class_selector.value
    )

    folder = CLASS_FOLDERS[
        selected_class
    ]

    images = get_images(folder)

    value = (
        image_input.value.strip()
    )

    if value == "":

        print(
            "Enter image number or filename."
        )

        return


    if value.isdigit():

        image_number = int(value)

        if (
            image_number < 1
            or
            image_number > len(images)
        ):

            print(
                "Image number does not exist."
            )

            return

        filename = images[
            image_number - 1
        ]

    else:

        filename = value

        if filename not in images:

            print(
                "Filename not found:",
                filename
            )

            return


    os.remove(
        os.path.join(
            folder,
            filename
        )
    )

    update_counts()

    print(
        "Deleted:",
        selected_class,
        filename
    )


delete_image_button.on_click(
    delete_specific_image
)


# ============================================================
# DISPLAY
# ============================================================

display(camera_view)

print("DRIVE")

display(
    widgets.HBox([
        start_button,
        stop_button,
        back_button
    ])
)

display(speed_slider)
display(steering)


print("MANUAL SAVE")

display(
    widgets.HBox([
        save_none_button,
        save_far_button,
        save_close_button
    ])
)


print("AUTO RECORD")

display(
    widgets.HBox([
        record_none_button,
        record_far_button,
        record_close_button,
        stop_record_button
    ])
)

display(record_status)


print("COUNTS")

display(
    widgets.HBox([
        none_count,
        far_count,
        close_count
    ])
)


print("DELETE")

display(class_selector)

display(delete_last_button)

display(
    widgets.HBox([
        from_box,
        to_box,
        delete_range_button
    ])
)

display(
    widgets.HBox([
        image_input,
        delete_image_button
    ])
)

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

DRIVE


FloatSlider(value=0.09, description='Speed:', max=0.3, min=0.05, step=0.01)

FloatSlider(value=0.0, description='Steering:', max=1.0, min=-1.0, step=0.01)

MANUAL SAVE


...

Label(value='Recording: OFF')

COUNTS


DELETE


Dropdown(description='Class:', options=('NONE', 'FAR', 'CLOSE'), value='NONE')

Button(description='DELETE LAST', style=ButtonStyle())

In [8]:
camera.stop()
print("Camera stopped")

Camera stopped


In [9]:
import os

none_dir = "datasets/stop/none"
far_dir = "datasets/stop/far"
close_dir = "datasets/stop/close"

def count_images(folder):
    return len([
        f for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

none_count = count_images(none_dir)
far_count = count_images(far_dir)
close_count = count_images(close_dir)

total = none_count + far_count + close_count

print("NONE :", none_count)
print("FAR  :", far_count)
print("CLOSE:", close_count)
print()
print("TOTAL:", total)

NONE : 81
FAR  : 189
CLOSE: 86

TOTAL: 356
